In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("/scratch/go76fil/Programs/Python/Paper_Topic_Modelling")
os.getcwd()

'/scratch/go76fil/Programs/Python/Paper_Topic_Modelling'

In [3]:
import pickle
with open('./output/store/pdf_files.pkl', 'rb') as f:
    pdf_files, year = pickle.load(f)
with open('./output/store/preprocessed_pdf_files.pkl', 'rb') as f:
    preprocessed_pdf_files, _ = pickle.load(f)

In [4]:
print(len(pdf_files))
print(len(preprocessed_pdf_files))

555
555


In [5]:
from src.optimisation import MTObjective 
from src.optimisation import basic_space  
from numpy.random import default_rng
seed = 42
objective = MTObjective(
    actual_docs=pdf_files,
    preprocessed_docs=preprocessed_pdf_files,
    analyzer="word",
    seed=seed,
    gpu=True
)
space = basic_space()

In [6]:
from hyperopt import fmin, tpe, Trials, space_eval

trials = Trials()
best = fmin(
    fn=objective,        
    space=space,           
    algo=tpe.suggest,      
    max_evals=20,          
    trials=trials,
    rstate=default_rng(seed)
)
best_hyperparams = space_eval(space, best)

print("Best hyperparameters:")
print(best_hyperparams)

Error with params {'embedding_model': 'allenai/scibert_scivocab_uncased', 'hdbscan_model': {'min_cluster_size': 30, 'min_samples': 19}, 'preprocess': True, 'umap_model': {'metric': 'cosine', 'min_dist': 0.760923897417212, 'n_components': 7, 'n_neighbors': 7}}: max_df corresponds to < documents than min_df
Error with params {'embedding_model': 'sentence-transformers/paraphrase-MiniLM-L3-v2', 'hdbscan_model': {'min_cluster_size': 13, 'min_samples': 18}, 'preprocess': False, 'umap_model': {'metric': 'cosine', 'min_dist': 0.6070032575501809, 'n_components': 5, 'n_neighbors': 26}}: max_df corresponds to < documents than min_df
100%|██████████████████████████████████████| 20/20 [09:34<00:00, 28.71s/trial, best loss: 0.09189189189189188]
Best hyperparameters:
{'embedding_model': 'allenai/scibert_scivocab_uncased', 'hdbscan_model': {'min_cluster_size': 16.0, 'min_samples': 20.0}, 'preprocess': False, 'umap_model': {'metric': 'cosine', 'min_dist': 0.08819450578021942, 'n_components': 4.0, 'n_ne

In [ ]:
with open('./output/store/optimal_hyperparams.pkl', 'wb') as f:
    pickle.dump((trials,best_hyperparams), f)